# Resonance Frequency as a Temporally Evolving Cardiac State During HRV Biofeedback
## Multi-Epoch Sliding-Window Analysis with Passive Control 

**Study design:** Two-group controlled, secondary analysis  
**Groups:** HRVBF (n=10, E01–E10) vs. Passive Control (n=10, K01–K10)  
**Temporal analyses:** Full session · First 5 min · Centre 5 min · Last 5 min  
**Primary outcome:** LFpeak/TP = LF_power / (LF_power + HF_power), bounded [0, 1]  
**Primary test:** Group × Condition interaction (LMM, REML)  



---
## Cell 1 — Imports and Configuration

In [ ]:
import warnings, os, shutil
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from scipy.signal import butter, filtfilt, find_peaks, welch
from scipy.interpolate import interp1d
from scipy.stats import wilcoxon, mannwhitneyu, beta as beta_dist, shapiro
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')

# ─── PATHS — edit these for your environment ───────────────────────────
HRVBF_DIR   = Path('/tmp/Source_HRVBF')
CONTROL_DIR = Path('/tmp/Source_Control')
OUT_DIR     = Path('/tmp/analysis_output')

FIG_PNG = OUT_DIR / 'figures/png'
FIG_EPS = OUT_DIR / 'figures/eps'
FIG_SVG = OUT_DIR / 'figures/svg'
TBL_DIR = OUT_DIR / 'tables'
for d in [FIG_PNG, FIG_EPS, FIG_SVG, TBL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── PARTICIPANT LISTS ──────────────────────────────────────────────────
HRVBF_IDS   = [f'E{i:02d}' for i in range(1, 11)]
CONTROL_IDS = [f'K{i:02d}' for i in range(1, 11)]

# ─── SIGNAL PARAMETERS ─────────────────────────────────────────────────
FS_PPG   = 128.0    # BVP sampling rate (Hz)
FS_RSP   = 32.0     # Respiration sampling rate (Hz)
FS_HRV   = 4.0      # IPI resampling rate (Hz)
WIN_LEN  = 60.0     # Sliding window length (s)
WIN_STEP = 30.0     # Step size — 50% overlap (s)
T1_DROP  = 60.0     # Stabilisation period discarded from BF/passive recordings (s)
SEG_LEN  = 300.0    # Standardised epoch length (s) = 5 min

LF_BAND  = (0.04, 0.15)
HF_BAND  = (0.15, 0.40)
RF_BAND  = (0.075, 0.115)
TP_BAND  = (0.04, 0.40)

MIN_RF_PROP       = 0.40   # LFpeak/TP threshold — cardiac RF criterion
MIN_RF_PROP_STRICT = 0.50  # Stricter threshold — cardiorespiratory RF criterion
MIN_RF_WINDOWS    = 3      # Minimum qualifying windows for RF-achieved
RF_RESP_LO        = 5.0   # brpm lower bound for cardiorespiratory criterion
RF_RESP_HI        = 9.0   # brpm upper bound

# ─── FIGURE STYLE — APB specifications ─────────────────────────────────
# Single column: 174 mm = 6.85 in
# Double column:  84 mm = 3.31 in
# Max height:    234 mm = 9.21 in
# Combination art: 600 dpi
# Font: Helvetica/Arial; 8–12 pt
W_SINGLE = 174 / 25.4   # inches
W_DOUBLE =  84 / 25.4   # inches
H_MAX    = 234 / 25.4   # inches
DPI      = 600

BLK, MID, LGT = '#1a1a1a', '#777777', '#cccccc'
BLUE, TEAL, AMBER = '#185fa5', '#0F6E56', '#BA7517'
FILL_H, FILL_C = '#3a3a3a', '#aaaaaa'

plt.rcParams.update({
    'font.family'       : 'Liberation Sans',
    'font.size'         : 9,
    'axes.labelsize'    : 9,
    'axes.titlesize'    : 9,
    'xtick.labelsize'   : 8,
    'ytick.labelsize'   : 8,
    'legend.fontsize'   : 8,
    'axes.linewidth'    : 0.5,
    'lines.linewidth'   : 0.9,
    'patch.linewidth'   : 0.5,
    'xtick.major.width' : 0.5,
    'ytick.major.width' : 0.5,
    'xtick.major.size'  : 2.5,
    'ytick.major.size'  : 2.5,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'savefig.dpi'       : DPI,
    'savefig.bbox'      : 'tight',
    'savefig.pad_inches': 0.03,
})

def save_figure(fig, stem):
    """Save figure in PNG, EPS, and SVG at APB specifications."""
    fig.savefig(FIG_PNG / f'{stem}.png', dpi=DPI, bbox_inches='tight', facecolor='white')
    fig.savefig(FIG_EPS / f'{stem}.eps', format='eps', bbox_inches='tight')
    fig.savefig(FIG_SVG / f'{stem}.svg', format='svg', bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved {stem}  [PNG 600dpi | EPS | SVG]')

print('Configuration loaded.')
print(f'  HRVBF_DIR  : {HRVBF_DIR}')
print(f'  CONTROL_DIR: {CONTROL_DIR}')
print(f'  Output     : {OUT_DIR}')

---
## Cell 2 — Signal Processing Functions

In [ ]:
# ── Filters ────────────────────────────────────────────────────────────
def bandpass(x, fs, lo, hi, order=4):
    b, a = butter(order, [lo/(fs/2), hi/(fs/2)], btype='bandpass')
    return filtfilt(b, a, x)

def lowpass(x, fs, hi, order=4):
    b, a = butter(order, hi/(fs/2), btype='lowpass')
    return filtfilt(b, a, x)

# ── IPI extraction from BVP/PPG ────────────────────────────────────────
def extract_ipi(ppg, fs):
    """
    Detrend -> bandpass 0.5-8 Hz -> peak detect (min 300 ms sep)
    -> artefact rejection (|IPI - median| > 30% or outside 300-2000 ms)
    -> cubic-spline interpolation over bad beats.
    Returns (t_ipi, ipi_corrected) or (None, None) on failure.
    """
    t = np.arange(len(ppg)) / fs
    ppg_d = ppg - np.polyval(np.polyfit(t, ppg, 1), t)
    ppg_f = bandpass(ppg_d, fs, 0.5, 8.0)
    peaks, _ = find_peaks(ppg_f, distance=int(0.30 * fs))
    if len(peaks) < 4:
        return None, None
    t_peaks = peaks / fs
    ipi = np.diff(t_peaks)
    t_ipi = t_peaks[1:]
    med = np.median(ipi)
    bad = (ipi < 0.30) | (ipi > 2.00) | (np.abs(ipi - med) > 0.30 * med)
    ipi_c = ipi.copy().astype(float)
    ipi_c[bad] = np.nan
    good = ~np.isnan(ipi_c)
    if good.sum() < 4:
        return None, None
    fi = interp1d(t_ipi[good], ipi_c[good], kind='cubic', fill_value='extrapolate')
    return t_ipi, fi(t_ipi)

def resample_ipi(t_ipi, ipi, fs_hrv=FS_HRV):
    """Resample IPI series to uniform fs_hrv Hz grid."""
    t_hrv = np.arange(t_ipi[0], t_ipi[-1], 1.0 / fs_hrv)
    return t_hrv, interp1d(t_ipi, ipi, kind='cubic', fill_value='extrapolate')(t_hrv)

# ── Spectral features ───────────────────────────────────────────────────
def band_power(f, Pxx, lo, hi):
    """Integrate Welch PSD over frequency band."""
    idx = (f >= lo) & (f <= hi)
    return float(np.trapezoid(Pxx[idx], f[idx])) if np.any(idx) else np.nan

def dominant_lf_freq(f, Pxx):
    """Return frequency of peak power in LF band."""
    idx = (f >= LF_BAND[0]) & (f <= LF_BAND[1])
    return float(f[idx][np.argmax(Pxx[idx])]) if np.any(idx) else np.nan

# ── Respiration rate ────────────────────────────────────────────────────
def resp_rate_bpm(rsp_seg, fs):
    """Estimate respiration rate (brpm) from chest belt segment via peak detection."""
    if len(rsp_seg) < fs * 5:
        return np.nan
    rsp_f = lowpass(rsp_seg, fs, 0.7)
    peaks, _ = find_peaks(rsp_f, distance=int(0.80 * fs))
    if len(peaks) < 2:
        return np.nan
    return float(60.0 / np.mean(np.diff(peaks / fs)))

# ── Clopper-Pearson 95% CI ──────────────────────────────────────────────
def clopper_pearson(k, n, alpha=0.05):
    lo = beta_dist.ppf(alpha/2, k, n-k+1) if k > 0 else 0.0
    hi = beta_dist.ppf(1-alpha/2, k+1, n-k) if k < n else 1.0
    return lo, hi

print('Signal processing functions defined.')

---
## Cell 3 — Data Loading Functions

In [ ]:
def load_baseline_txt(path):
    """
    Parse BioTrace+ raw export TXT.
    Col 1: BVP at 128 Hz. Col 2: RSP stored at 128 Hz (repeating 32-Hz values -> decimate 4x).
    Returns (bvp_array, rsp_array).
    """
    with open(path, 'r', errors='replace') as f:
        lines = f.readlines()
    hdr = next(i for i, l in enumerate(lines) if l.strip().startswith('TIME'))
    bvp, rsp = [], []
    for l in lines[hdr + 2:]:
        l = l.strip()
        if not l:
            continue
        parts = l.replace('\t', ',').split(',')
        if len(parts) < 3:
            continue
        try:
            bvp.append(float(parts[1]))
            rsp.append(float(parts[2]))
        except ValueError:
            continue
    return np.array(bvp, float), np.array(rsp, float)[::4]  # decimate RSP to 32 Hz


def load_edf(path):
    """
    Load BF/passive EDF (Nexus-10 MKII / BioTrace+).
    Channels: 'Sensor-G:BVP' (128 Hz), 'Sensor-H:RSP' (stored at 128 Hz, decimated to 32 Hz).
    Returns (bvp, rsp, fs_bvp).
    """
    import mne
    mne.set_log_level('ERROR')
    raw = mne.io.read_raw_edf(str(path), preload=True, verbose=False)
    fs = float(raw.info['sfreq'])
    bvp = raw.get_data(picks='Sensor-G:BVP').flatten()
    rsp = raw.get_data(picks='Sensor-H:RSP').flatten()[::4]  # decimate to 32 Hz
    return bvp, rsp, fs


def locate_baseline(participant_id, source_dir):
    """Return Path to baseline TXT for a participant, or None."""
    pid = participant_id
    # Exact pattern first
    exact = list(source_dir.glob(f'{pid}_Baseline*.txt'))
    if exact:
        return exact[0]
    # Control group bracket-style names, e.g. K01_02[Baseline].txt
    bracket = [f for f in source_dir.iterdir()
               if f.suffix == '.txt'
               and pid.lower() in f.name.lower()
               and ('aseline' in f.name or 'Base' in f.name)]
    if bracket:
        return bracket[0]
    # Any TXT starting with the participant ID
    fallback = [f for f in source_dir.iterdir()
                if f.suffix == '.txt' and f.name.upper().startswith(pid.upper())]
    if fallback:
        return fallback[0]
    # HRVBF legacy 'ffBase' filenames (numbered order)
    ff = sorted(source_dir.glob('ffBase*.txt'))
    idx = int(pid[1:]) - 1
    if idx < len(ff):
        return ff[idx]
    return None


print('Data loading functions defined.')

---
## Cell 4 — Temporal Epoch Framework + Window Extraction Engine

In [ ]:
def extract_segment(bvp, rsp, fs_bvp, fs_rsp, drop_s=0.0,
                     epoch='full', seg_len=SEG_LEN):
    """
    After discarding `drop_s` seconds of stabilisation, extract:
      'full'   — all remaining signal
      'first'  — first seg_len seconds
      'center' — middle seg_len seconds
      'last'   — last seg_len seconds
    """
    d_b = int(drop_s * fs_bvp)
    d_r = int(drop_s * fs_rsp)
    bvp_s = bvp[d_b:] if len(bvp) > d_b else bvp
    rsp_s = rsp[d_r:] if len(rsp) > d_r else rsp
    nb, nr = int(seg_len * fs_bvp), int(seg_len * fs_rsp)
    if epoch == 'full':
        return bvp_s, rsp_s
    elif epoch == 'first':
        return bvp_s[:nb], rsp_s[:nr]
    elif epoch == 'center':
        mb, mr = len(bvp_s)//2, len(rsp_s)//2
        return bvp_s[max(0,mb-nb//2):max(0,mb-nb//2)+nb], \
               rsp_s[max(0,mr-nr//2):max(0,mr-nr//2)+nr]
    elif epoch == 'last':
        return (bvp_s[-nb:] if len(bvp_s) >= nb else bvp_s), \
               (rsp_s[-nr:] if len(rsp_s) >= nr else rsp_s)
    return bvp_s, rsp_s


def run_sliding_windows(bvp, rsp, fs_bvp, fs_rsp, participant_id, phase):
    """
    Core window analysis.
    Primary metric: LFpeak_TP = LF_power / (LF_power + HF_power)  [0, 1]
    RF criteria:
      Cardiac RF  : f_peak in RF_BAND AND LFpeak_TP >= MIN_RF_PROP
      CR-RF       : also requires RF_RESP_LO <= resp <= RF_RESP_HI
    """
    t_ipi, ipi = extract_ipi(bvp, fs_bvp)
    if t_ipi is None:
        return []
    t_hrv, hrv = resample_ipi(t_ipi, ipi, FS_HRV)
    t_rsp = np.arange(len(rsp)) / fs_rsp
    rows = []
    for ws in np.arange(t_hrv[0], t_hrv[-1] - WIN_LEN + 1e-6, WIN_STEP):
        we = ws + WIN_LEN
        idx_h = (t_hrv >= ws) & (t_hrv < we)
        if idx_h.sum() < FS_HRV * WIN_LEN * 0.80:
            continue
        f, Pxx = welch(hrv[idx_h], FS_HRV, nperseg=int(idx_h.sum()))
        lf  = band_power(f, Pxx, *LF_BAND)
        hf  = band_power(f, Pxx, *HF_BAND)
        tp  = band_power(f, Pxx, *TP_BAND)
        fpk = dominant_lf_freq(f, Pxx)
        lf_tp   = (lf / (lf + hf)) if (hf > 0 and not np.isnan(lf)) else np.nan
        lf_hf   = (lf / hf) if hf > 0 else np.nan
        idx_r   = (t_rsp >= ws) & (t_rsp < we)
        rr      = resp_rate_bpm(rsp[idx_r], fs_rsp) if idx_r.sum() > 0 else np.nan
        in_band = not np.isnan(fpk) and RF_BAND[0] <= fpk <= RF_BAND[1]
        rf_c    = int(in_band and not np.isnan(lf_tp) and lf_tp >= MIN_RF_PROP)
        rf_cr   = int(in_band and not np.isnan(lf_tp) and lf_tp >= MIN_RF_PROP_STRICT
                      and not np.isnan(rr) and RF_RESP_LO <= rr <= RF_RESP_HI)
        rows.append({
            'Participant'   : participant_id,
            'Group'         : 'HRVBF' if participant_id.startswith('E') else 'Control',
            'Phase'         : phase,
            'Condition'     : 0 if phase == 'Baseline' else 1,
            'GroupCode'     : 1 if participant_id.startswith('E') else 0,
            'WindowIndex'   : len(rows),
            'WindowStart_s' : float(ws),
            'WindowEnd_s'   : float(we),
            'f_peak_Hz'     : fpk,
            'LFpeak_TP'     : lf_tp,     # PRIMARY METRIC [0, 1]
            'LF_power'      : lf,
            'HF_power'      : hf,
            'TP'            : tp,
            'LF_HF'         : lf_hf,
            'RespRate_bpm'  : rr,
            'RF_cardiac'    : rf_c,
            'RF_CR'         : rf_cr,
        })
    return rows


EPOCHS = {
    'full'  : 'Full session',
    'first' : 'First 5 min',
    'center': 'Centre 5 min',
    'last'  : 'Last 5 min',
}

print('Temporal epoch framework defined.')
print('Epochs:', list(EPOCHS.values()))

---
## Cell 5 — Run All Four Analyses

In [ ]:
import mne; mne.set_log_level('ERROR')

# epoch_data[epoch] = DataFrame of all windows for that epoch
epoch_data = {}

for epoch_key, epoch_label in EPOCHS.items():
    print(f'\n== {epoch_label} ==')
    all_rows = []

    for pid in HRVBF_IDS:
        src = HRVBF_DIR
        drop = 0.0 if False else 0.0  # baseline: no drop

        # Baseline
        bp = locate_baseline(pid, src)
        if bp:
            bvp_b, rsp_b = load_baseline_txt(bp)
            bseg, rseg = extract_segment(bvp_b, rsp_b, FS_PPG, FS_RSP, 0.0, epoch_key)
            all_rows.extend(run_sliding_windows(bseg, rseg, FS_PPG, FS_RSP, pid, 'Baseline'))

        # Intervention
        edf = src / f'{pid}_BF.edf'
        if edf.exists():
            bvp_i, rsp_i, fs_i = load_edf(edf)
            bseg, rseg = extract_segment(bvp_i, rsp_i, fs_i, FS_RSP, T1_DROP, epoch_key)
            all_rows.extend(run_sliding_windows(bseg, rseg, fs_i, FS_RSP, pid, 'Intervention'))

    for pid in CONTROL_IDS:
        src = CONTROL_DIR

        # Baseline
        bp = locate_baseline(pid, src)
        if bp:
            bvp_b, rsp_b = load_baseline_txt(bp)
            bseg, rseg = extract_segment(bvp_b, rsp_b, FS_PPG, FS_RSP, 0.0, epoch_key)
            all_rows.extend(run_sliding_windows(bseg, rseg, FS_PPG, FS_RSP, pid, 'Baseline'))

        # Passive sitting
        edf = src / f'{pid}_BF.edf'
        if edf.exists():
            bvp_i, rsp_i, fs_i = load_edf(edf)
            bseg, rseg = extract_segment(bvp_i, rsp_i, fs_i, FS_RSP, T1_DROP, epoch_key)
            all_rows.extend(run_sliding_windows(bseg, rseg, fs_i, FS_RSP, pid, 'Intervention'))

    df = pd.DataFrame(all_rows)
    epoch_data[epoch_key] = df
    print(f'  Total windows: {len(df)}')
    print(df.groupby(['Group','Phase'])['Participant'].count().to_string())

# Save window-level data for all epochs
for ek, df in epoch_data.items():
    df.to_csv(TBL_DIR / f'windows_{ek}.csv', index=False)
print('\nAll window-level CSVs saved.')

---
## Cell 6 — Participant-Level Summaries

In [ ]:
def participant_summary(df_win, pid, phase):
    sub = df_win[(df_win.Participant == pid) & (df_win.Phase == phase)]
    if sub.empty:
        return None
    n_rf  = int(sub.RF_cardiac.sum())
    n_cr  = int(sub.RF_CR.sum())
    n_win = len(sub)
    rf_achieved    = n_rf >= MIN_RF_WINDOWS
    cr_achieved    = n_cr >= MIN_RF_WINDOWS
    rf_rows = sub[sub.RF_cardiac == 1].sort_values('WindowStart_s')
    # Union-of-intervals RF duration (no double-counting of overlapping windows)
    union_s = 0.0
    prev_end = -np.inf
    for _, row in rf_rows.iterrows():
        s, e = row.WindowStart_s, row.WindowEnd_s
        union_s += (e - s) if s >= prev_end else max(0.0, e - prev_end)
        prev_end = max(prev_end, e)
    total_s = sub.WindowEnd_s.max() - sub.WindowStart_s.min()
    rf_pct  = 100.0 * union_s / total_s if total_s > 0 else np.nan
    onset_s = float(rf_rows.WindowStart_s.iloc[0]) if (rf_achieved and not rf_rows.empty) else np.nan
    return {
        'Participant'      : pid,
        'Group'            : sub.Group.iloc[0],
        'Phase'            : phase,
        'n_windows'        : n_win,
        'n_RF_cardiac'     : n_rf,
        'n_RF_CR'          : n_cr,
        'RF_achieved'      : rf_achieved,
        'RF_CR_achieved'   : cr_achieved,
        'RF_duration_min'  : union_s / 60.0,
        'RF_percent'       : rf_pct,
        'Onset_s'          : onset_s,
        'Median_f_peak'    : float(rf_rows.f_peak_Hz.median()) if not rf_rows.empty else np.nan,
        'Median_LFpeak_TP' : float(rf_rows.LFpeak_TP.median()) if not rf_rows.empty else np.nan,
        'Mean_Resp'        : float(sub.RespRate_bpm.mean(skipna=True)),
    }


# Build participant-level summaries for all epochs
# Full-session summary is used for Figs 2-3, 6-7 and Tables 1-5
part_data = {}
for ek in EPOCHS:
    df_win = epoch_data[ek]
    rows = []
    for pid in HRVBF_IDS + CONTROL_IDS:
        for phase in ['Baseline', 'Intervention']:
            r = participant_summary(df_win, pid, phase)
            if r:
                rows.append(r)
    part_data[ek] = pd.DataFrame(rows)
    part_data[ek].to_csv(TBL_DIR / f'participants_{ek}.csv', index=False)

# Convenience references
df_win_full = epoch_data['full']
df_part     = part_data['full']

print('Participant summaries computed for all epochs.')
print(df_part[['Participant','Group','Phase','n_RF_cardiac','RF_achieved',
               'RF_duration_min','Mean_Resp']].to_string(index=False))

---
## Cell 7 — LMM: Group × Condition for All Four Epochs

In [ ]:
def run_lmm(df_win):
    """Fit Group x Condition LMM on LFpeak_TP; return result dict."""
    sub = df_win[['Participant','GroupCode','Condition','LFpeak_TP']].dropna()
    if len(sub) < 20:
        return None
    model  = smf.mixedlm('LFpeak_TP ~ Condition * GroupCode', sub,
                          groups=sub['Participant'])
    result = model.fit(reml=True, method='nm')
    tau2   = float(result.cov_re.iloc[0, 0])
    sig2   = float(result.scale)
    icc    = tau2 / (tau2 + sig2) if (tau2 + sig2) > 0 else 0.0
    n_per  = sub.groupby('Participant').size().mean()
    n_eff  = len(sub) / (1 + (n_per - 1) * icc) if icc > 0 else len(sub)
    fe = result.fe_params
    ci = result.conf_int()
    pv = result.pvalues
    inter = [t for t in fe.index if 'Condition' in t and 'GroupCode' in t]
    cond  = [t for t in fe.index if t == 'Condition']
    grp   = [t for t in fe.index if t == 'GroupCode']
    return {
        'n_windows'  : len(sub),
        'icc'        : round(icc, 3),
        'n_eff'      : round(n_eff, 1),
        'beta_inter' : round(fe[inter[0]], 4) if inter else np.nan,
        'ci_lo_inter': round(ci.loc[inter[0], 0], 4) if inter else np.nan,
        'ci_hi_inter': round(ci.loc[inter[0], 1], 4) if inter else np.nan,
        'p_inter'    : round(pv[inter[0]], 4) if inter else np.nan,
        'beta_cond'  : round(fe[cond[0]], 4) if cond else np.nan,
        'p_cond'     : round(pv[cond[0]], 4) if cond else np.nan,
        'beta_grp'   : round(fe[grp[0]], 4) if grp else np.nan,
        'p_grp'      : round(pv[grp[0]], 4) if grp else np.nan,
        'result'     : result,
    }


lmm_results = {}
for ek, label in EPOCHS.items():
    r = run_lmm(epoch_data[ek])
    lmm_results[ek] = r
    if r:
        sig = '*' if r['p_inter'] < 0.05 else ''
        print(f'{label:20s}: beta={r["beta_inter"]:.3f} '
              f'[{r["ci_lo_inter"]:.3f},{r["ci_hi_inter"]:.3f}] '
              f'p={r["p_inter"]:.3f}{sig}  ICC={r["icc"]:.3f}  Neff={r["n_eff"]:.0f}  '
              f'N_win={r["n_windows"]}')

---
## Cell 8 — Tables 1–6 

In [ ]:
def fmt(v, d=2): return '—' if pd.isna(v) else f'{v:.{d}f}'
def pct_str(v): return '—' if pd.isna(v) else f'{v:.1f}'
def iqr_str(vals, d=2):
    v = np.array(vals, float)
    v = v[~np.isnan(v)]
    if len(v) == 0: return '—'
    return f'{np.median(v):.{d}f} [{np.percentile(v,25):.{d}f}–{np.percentile(v,75):.{d}f}]'
def cp_str(k, n):
    lo, hi = clopper_pearson(k, n)
    return f'{100*k/n:.0f}% [{100*lo:.0f}–{100*hi:.0f}%]'

def wilcox_row(grp_b, grp_i, var, label, group):
    m = grp_b[['Participant', var]].merge(grp_i[['Participant', var]],
        on='Participant', suffixes=('_b','_i')).dropna()
    if len(m) < 3:
        return {'Group':group,'Outcome':label,'N':len(m),'Baseline':'-','Intervention':'-','W':'-','p':'-','r':'-'}
    bv, iv = m[f'{var}_b'].values, m[f'{var}_i'].values
    stat, p = wilcoxon(bv, iv, alternative='two-sided', zero_method='wilcox')
    N = len(m)
    Z = (stat - N*(N+1)/4) / np.sqrt(N*(N+1)*(2*N+1)/24)
    r = abs(Z) / np.sqrt(N)
    return {'Group':group,'Outcome':label,'N':N,'Baseline':iqr_str(bv),'Intervention':iqr_str(iv),
            'W':int(stat),'p':f'{p:.3f}'+('*' if p<.05 else ''),'r':f'{r:.2f}'}

def mwu_row(bf_vals, ct_vals, var, label):
    bfv = np.array(bf_vals, float); bfv = bfv[~np.isnan(bfv)]
    ctv = np.array(ct_vals, float); ctv = ctv[~np.isnan(ctv)]
    if len(bfv) < 2 or len(ctv) < 2:
        return {'Outcome':label,'HRVBF':'-','Control':'-','U':'-','p':'-','r':'-'}
    stat, p = mannwhitneyu(bfv, ctv, alternative='two-sided')
    N = len(bfv) + len(ctv)
    Z = (stat - len(bfv)*len(ctv)/2) / np.sqrt(len(bfv)*len(ctv)*(N+1)/12)
    r = abs(Z) / np.sqrt(N)
    return {'Outcome':label,'HRVBF':iqr_str(bfv),'Control':iqr_str(ctv),
            'U':int(stat),'p':f'{p:.3f}'+('*' if p<.05 else ''),'r':f'{r:.2f}'}

vars_lbl = [
    ('RF_duration_min', 'RF Duration (min)'),
    ('RF_percent',      'RF Coverage (%)'),
    ('Mean_Resp',       'Respiration Rate (brpm)'),
    ('Median_LFpeak_TP','Median LFpeak/TP'),
]

# ── TABLE 1: HRVBF group — full-session intervention metrics ───────────
bf_i = df_part[(df_part.Group=='HRVBF') & (df_part.Phase=='Intervention')].sort_values('Participant')
t1 = []
for _, r in bf_i.iterrows():
    t1.append({'Participant':r.Participant,'RF Achieved':('Yes' if r.RF_achieved else 'No'),
        'N Windows':int(r.n_windows),'RF Windows':int(r.n_RF_cardiac),
        'RF Duration (min)':fmt(r.RF_duration_min,1),'RF Coverage (%)':pct_str(r.RF_percent),
        'Onset Latency (s)':fmt(r.Onset_s,0),'Median f-peak (Hz)':fmt(r.Median_f_peak,3),
        'Median LFpeak/TP':fmt(r.Median_LFpeak_TP,3),'Mean Resp (brpm)':fmt(r.Mean_Resp,1),
        'CR-RF Achieved':('Yes' if r.RF_CR_achieved else 'No')})
nrf, n = bf_i.RF_achieved.sum(), len(bf_i)
t1.append({'Participant':f'Summary (n={n})','RF Achieved':cp_str(nrf,n),
    'N Windows':f"{bf_i.n_windows.sum()} total",'RF Windows':f"{bf_i.n_RF_cardiac.sum()} total",
    'RF Duration (min)':f"Mdn {fmt(bf_i.RF_duration_min.median(),1)}",
    'RF Coverage (%)':f"Mdn {pct_str(bf_i.RF_percent.median())}",
    'Onset Latency (s)':f"Mdn {fmt(bf_i.Onset_s.median(),0)}",
    'Median f-peak (Hz)':f"Mdn {fmt(bf_i.Median_f_peak.median(),3)}",
    'Median LFpeak/TP':f"Mdn {fmt(bf_i.Median_LFpeak_TP.median(),3)}",
    'Mean Resp (brpm)':f"Mdn {fmt(bf_i.Mean_Resp.median(),1)}",
    'CR-RF Achieved':cp_str(bf_i.RF_CR_achieved.sum(),n)})
df_t1 = pd.DataFrame(t1)
df_t1.to_csv(TBL_DIR/'Table1_HRVBF_Intervention.csv', index=False)
t1_caption = (
    'Table 1. HRVBF group (n = 10): individual cardiac RF metrics during the full-session '
    'paced-breathing phase (T2, after 60-s stabilisation discard). RF windows = windows '
    'meeting the cardiac criterion (f_peak in [0.075, 0.115] Hz; LFpeak/TP >= 0.40). '
    'RF duration = union of qualifying window intervals (overlapping segments not double-counted). '
    'Onset latency = start time of the first RF-qualifying window (s from T2 onset). '
    'Median f-peak and LFpeak/TP computed across RF windows only. '
    'CR-RF = cardiorespiratory criterion (additionally LFpeak/TP >= 0.50 and 5-9 brpm). '
    'Prevalence cells show k/n (%, 95% Clopper-Pearson CI). Mdn = group median.'
)
with open(TBL_DIR/'Table1_caption.txt','w') as f: f.write(t1_caption)
print('Table 1 saved.\n', df_t1.to_string(index=False))

# ── TABLE 2: Control group — full-session passive metrics ──────────────
ct_i = df_part[(df_part.Group=='Control') & (df_part.Phase=='Intervention')].sort_values('Participant')
t2 = []
for _, r in ct_i.iterrows():
    t2.append({'Participant':r.Participant,'N Windows':int(r.n_windows),'RF Windows':int(r.n_RF_cardiac),
        'RF Achieved':('Yes' if r.RF_achieved else 'No'),
        'RF Duration (min)':fmt(r.RF_duration_min,1),'RF Coverage (%)':pct_str(r.RF_percent),
        'Median f-peak (Hz)':fmt(r.Median_f_peak,3),'Median LFpeak/TP':fmt(r.Median_LFpeak_TP,3),
        'Mean Resp (brpm)':fmt(r.Mean_Resp,1)})
nrfc, nc = ct_i.RF_achieved.sum(), len(ct_i)
t2.append({'Participant':f'Summary (n={nc})','N Windows':f"{ct_i.n_windows.sum()} total",
    'RF Windows':f"{ct_i.n_RF_cardiac.sum()} total",'RF Achieved':cp_str(nrfc,nc),
    'RF Duration (min)':f"Mdn {fmt(ct_i.RF_duration_min.median(),1)}",
    'RF Coverage (%)':f"Mdn {pct_str(ct_i.RF_percent.median())}",
    'Median f-peak (Hz)':f"Mdn {fmt(ct_i.Median_f_peak.median(),3)}",
    'Median LFpeak/TP':f"Mdn {fmt(ct_i.Median_LFpeak_TP.median(),3)}",
    'Mean Resp (brpm)':f"Mdn {fmt(ct_i.Mean_Resp.median(),1)}"})
df_t2 = pd.DataFrame(t2)
df_t2.to_csv(TBL_DIR/'Table2_Control_Passive.csv', index=False)
t2_caption = (
    'Table 2. Control group (n = 10): individual cardiac RF metrics during the full-session '
    'passive-sitting phase (T2, after 60-s stabilisation discard). Same classification criteria '
    'as Table 1. No control participant met the cardiorespiratory RF criterion during any phase. '
    'Prevalence cells show k/n (%, 95% Clopper-Pearson CI).'
)
with open(TBL_DIR/'Table2_caption.txt','w') as f: f.write(t2_caption)
print('\nTable 2 saved.')

# ── TABLE 3: Within-group Wilcoxon ─────────────────────────────────────
t3 = []
for grp in ['HRVBF', 'Control']:
    gb = df_part[(df_part.Group==grp) & (df_part.Phase=='Baseline')]
    gi = df_part[(df_part.Group==grp) & (df_part.Phase=='Intervention')]
    for var, lbl in vars_lbl:
        t3.append(wilcox_row(gb, gi, var, lbl, grp))
df_t3 = pd.DataFrame(t3)
df_t3.to_csv(TBL_DIR/'Table3_Wilcoxon_within.csv', index=False)
t3_caption = (
    'Table 3. Within-group Wilcoxon signed-rank tests comparing participant-level cardiac RF '
    'metrics between spontaneous-breathing baseline and the intervention phase (full-session '
    'analysis). Values: median [IQR, 25th-75th percentile]. '
    'W = Wilcoxon signed-rank statistic; r = Z/sqrt(N) effect size. * p < .05. '
    'K02 excluded from control baseline comparisons (baseline recording unavailable). '
    'brpm = breaths per minute.'
)
with open(TBL_DIR/'Table3_caption.txt','w') as f: f.write(t3_caption)
print('\nTable 3 saved.\n', df_t3.to_string(index=False))

# ── TABLE 4: Between-group Mann-Whitney U ──────────────────────────────
t4 = []
for var, lbl in vars_lbl:
    t4.append(mwu_row(bf_i[var].values, ct_i[var].values, var, lbl))
df_t4 = pd.DataFrame(t4)
df_t4.to_csv(TBL_DIR/'Table4_MWU_between.csv', index=False)
t4_caption = (
    'Table 4. Between-group Mann-Whitney U tests comparing intervention-phase '
    'participant-level metrics between the HRVBF and passive-control groups '
    '(full-session analysis). Values: median [IQR]. '
    'U = Mann-Whitney statistic; r = Z/sqrt(N) effect size. * p < .05.'
)
with open(TBL_DIR/'Table4_caption.txt','w') as f: f.write(t4_caption)
print('\nTable 4 saved.\n', df_t4.to_string(index=False))

# ── TABLE 5: LMM fixed effects — full session ──────────────────────────
r5 = lmm_results['full']
if r5:
    result5 = r5['result']
    fe = result5.fe_params; ci = result5.conf_int(); pv = result5.pvalues
    labels5 = {'Intercept':'Intercept (Control, Baseline)',
                'Condition':'Condition (Intervention vs Baseline, Control ref.)',
                'GroupCode':'Group (HRVBF vs Control, Baseline)',
                'Condition:GroupCode':'Group x Condition interaction'}
    t5 = [{'Term':labels5.get(t,t),'beta':round(fe[t],4),
            '95% CI Lo':round(ci.loc[t,0],4),'95% CI Hi':round(ci.loc[t,1],4),
            'p':round(pv[t],4),'Sig':'*' if pv[t]<0.05 else ''}
           for t in fe.index]
    t5.append({'Term':'ICC','beta':r5['icc'],'95% CI Lo':'','95% CI Hi':'','p':'','Sig':''})
    t5.append({'Term':'Effective N','beta':r5['n_eff'],'95% CI Lo':'','95% CI Hi':'','p':'','Sig':''})
    t5.append({'Term':'Total windows','beta':r5['n_windows'],'95% CI Lo':'','95% CI Hi':'','p':'','Sig':''})
    df_t5 = pd.DataFrame(t5)
    df_t5.to_csv(TBL_DIR/'Table5_LMM_fullsession.csv', index=False)
    t5_caption = (
        'Table 5. Linear mixed-effects model (REML) fixed-effect estimates for the full-session '
        'analysis. Outcome: LFpeak/TP = LF power / (LF power + HF power), range [0, 1]. '
        'Fixed effects: Group (HRVBF = 1, Control = 0), Condition (Intervention = 1, Baseline = 0), '
        'and their interaction; random intercept per participant. Reference cell: Control-Baseline. '
        'ICC = intraclass correlation coefficient (between-participant variance / total variance). '
        'Effective N computed from Kish formula. Confidence intervals and p-values are Wald-based. '
        '* p < .05.'
    )
    with open(TBL_DIR/'Table5_caption.txt','w') as f: f.write(t5_caption)
    print('\nTable 5 saved.\n', df_t5.to_string(index=False))

# ── TABLE 6: Multi-epoch comparison ────────────────────────────────────
def prevalence_str(df_win, group, phase):
    pids = df_win[(df_win.Group==group)&(df_win.Phase==phase)].Participant.unique()
    achieved = sum(
        df_win[(df_win.Participant==pid)&(df_win.Phase==phase)].RF_cardiac.sum() >= MIN_RF_WINDOWS
        for pid in pids)
    n = len(pids)
    return f'{achieved}/{n}  ({100*achieved/n:.0f}%)' if n > 0 else '—'

def wilcox_rf_dur(df_part_ek, group):
    gb = df_part_ek[(df_part_ek.Group==group)&(df_part_ek.Phase=='Baseline')].set_index('Participant')
    gi = df_part_ek[(df_part_ek.Group==group)&(df_part_ek.Phase=='Intervention')].set_index('Participant')
    common = gb.index.intersection(gi.index)
    if len(common) < 3: return '-','-','-'
    bv = gb.loc[common,'RF_duration_min'].values
    iv = gi.loc[common,'RF_duration_min'].values
    stat,p = wilcoxon(bv,iv,alternative='two-sided',zero_method='wilcox')
    N=len(common); Z=(stat-N*(N+1)/4)/np.sqrt(N*(N+1)*(2*N+1)/24); r=abs(Z)/np.sqrt(N)
    return int(stat), f'{p:.3f}'+('*' if p<.05 else ''), f'{r:.2f}'

t6 = []
for ek, label in EPOCHS.items():
    lr = lmm_results[ek]
    dw = epoch_data[ek]
    dp = part_data[ek]
    W, p_str, r_str = wilcox_rf_dur(dp, 'HRVBF')
    row = {
        'Temporal Analysis'         : label,
        'N Windows (total)'         : len(dw) if lr else '—',
        'Group x Cond beta'         : lr['beta_inter'] if lr else '—',
        '95% CI Lo'                 : lr['ci_lo_inter'] if lr else '—',
        '95% CI Hi'                 : lr['ci_hi_inter'] if lr else '—',
        'p'                         : (f"{lr['p_inter']:.3f}" + ('*' if lr['p_inter'] < 0.05 else '')) if lr else '—',
        'ICC'                       : lr['icc'] if lr else '—',
        'N_eff'                     : lr['n_eff'] if lr else '—',
        'HRVBF RF prev. Baseline'   : prevalence_str(dw,'HRVBF','Baseline'),
        'HRVBF RF prev. Intervention': prevalence_str(dw,'HRVBF','Intervention'),
        'Control RF prev. Baseline' : prevalence_str(dw,'Control','Baseline'),
        'Control RF prev. Passive'  : prevalence_str(dw,'Control','Intervention'),
        'HRVBF W'                   : W,
        'HRVBF Wilcoxon p'          : p_str,
        'HRVBF r'                   : r_str,
        'HRVBF Resp Base (brpm)'    : fmt(dp[(dp.Group=='HRVBF')&(dp.Phase=='Baseline')]['Mean_Resp'].median(),1),
        'HRVBF Resp Interv (brpm)'  : fmt(dp[(dp.Group=='HRVBF')&(dp.Phase=='Intervention')]['Mean_Resp'].median(),1),
        'Control Resp Base (brpm)'  : fmt(dp[(dp.Group=='Control')&(dp.Phase=='Baseline')]['Mean_Resp'].median(),1),
        'Control Resp Passive (brpm)': fmt(dp[(dp.Group=='Control')&(dp.Phase=='Intervention')]['Mean_Resp'].median(),1),
    }
    t6.append(row)
df_t6 = pd.DataFrame(t6)
df_t6.to_csv(TBL_DIR/'Table6_MultiEpoch.csv', index=False)
t6_caption = (
    'Table 6. Multi-epoch comparison of Group x Condition LMM results across four temporal analyses. '
    'LFpeak/TP = LF_power / (LF_power + HF_power), range [0, 1]. '
    'beta = Group x Condition interaction fixed effect (REML); 95% CI = Wald confidence interval. '
    'ICC = intraclass correlation (between-participant / total variance). '
    'N_eff = effective sample size (Kish formula). '
    'RF prevalence = proportion of participants meeting cardiac RF criterion '
    '(>= 3 windows with f_peak in [0.075, 0.115] Hz and LFpeak/TP >= 0.40). '
    'Wilcoxon W and r = within-group RF duration change (Baseline -> Intervention) in the HRVBF group. '
    'Resp = median window-level respiration rate (brpm). * p < .05.'
)
with open(TBL_DIR/'Table6_caption.txt','w') as f: f.write(t6_caption)
print('\nTable 6 saved.\n', df_t6[['Temporal Analysis','N Windows (total)','Group x Cond beta','p','ICC','N_eff']].to_string(index=False))

---
## Cell 9 — Figure 1: Study Protocol Diagram

In [ ]:
fig = plt.figure(figsize=(W_SINGLE, 3.5), facecolor='white')
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, 10); ax.set_ylim(0, 5); ax.axis('off')

def rbox(ax, x, y, w, h, txt1, txt2='', fc='#f4f4f4', ec='#555', fs1=8, fs2=7):
    r = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                        fc=fc, ec=ec, lw=0.5, zorder=3)
    ax.add_patch(r)
    if txt2:
        ax.text(x+w/2, y+h*0.62, txt1, ha='center', va='center',
                fontsize=fs1, fontweight='bold', color=BLK, zorder=4)
        ax.text(x+w/2, y+h*0.28, txt2, ha='center', va='center',
                fontsize=fs2, color=MID, zorder=4)
    else:
        ax.text(x+w/2, y+h/2, txt1, ha='center', va='center',
                fontsize=fs1, fontweight='bold', color=BLK, zorder=4)

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='-|>', color=MID, lw=0.6))

# HRVBF group
ax.text(0.1, 4.7, 'HRVBF group  (n = 10)', fontsize=8, fontweight='bold', color=BLK)
rbox(ax, 0.1, 3.5, 2.5, 0.9, 'Baseline (T0)', 'Spontaneous breathing  ~5 min')
arrow(ax, 1.35, 3.5, 1.35, 3.0)
rbox(ax, 0.1, 2.1, 2.5, 1.1, 'Paced breathing (T2)', '0.1 Hz visual biofeedback  ~10 min',
     fc='#ddeeff', ec=BLUE)

# Divider
ax.plot([0.1, 9.9], [1.8, 1.8], color='#cccccc', lw=0.5, ls='--')

# Control group
ax.text(0.1, 1.65, 'Control group  (n = 10)', fontsize=8, fontweight='bold', color=BLK)
rbox(ax, 0.1, 0.55, 2.5, 0.9, 'Baseline (T0)', 'Spontaneous breathing  ~5 min')
arrow(ax, 1.35, 0.55, 1.35, 0.0)
rbox(ax, 0.1, -0.9, 2.5, 0.9, 'Passive sitting (T2)', 'Spontaneous breathing  ~10 min')

# Epoch framework panel
rbox(ax, 3.2, 3.7, 6.6, 0.65, 'Full session  (~9 min after 60-s stabilisation discard)',
     fc='#e8e8e8', ec='#888')
epoch_specs = [
    (3.2,  2.85, 2.0, 0.55, 'First 5 min', '#d5e8f5', BLUE),
    (5.3,  2.85, 2.0, 0.55, 'Centre 5 min', '#d5f0ea', TEAL),
    (7.4,  2.85, 2.0, 0.55, 'Last 5 min', '#fce8c0', AMBER),
]
for xs, ys, w, h, lbl, fc, ec in epoch_specs:
    rbox(ax, xs, ys, w, h, lbl, fc=fc, ec=ec)

# Window tiles
tile_xs = np.linspace(3.3, 9.5, 10)
for i, xi in enumerate(tile_xs):
    r = FancyBboxPatch((xi, 2.1), 0.55, 0.5, boxstyle='round,pad=0.04',
                        fc=BLUE, ec='none', alpha=0.3 + 0.4*(i%2))
    ax.add_patch(r)
ax.text(6.5, 1.9, '60-s windows (50% overlap)', ha='center', fontsize=7, color=MID)

# LMM box
rbox(ax, 3.2, 0.6, 6.6, 0.9,
     'LMM: Group x Condition   (x 4 analyses)',
     'Primary test: beta_interaction [95% CI], p, ICC, N_eff',
     fc='#ede8f8', ec='#534AB7')

# Signal annotation
ax.text(0.1, -1.15, 'Both groups: PPG + Respiration -> IPI extraction -> Welch PSD -> LFpeak/TP = LF/(LF+HF)  [0,1]',
        fontsize=6.5, color=MID)

fig.tight_layout()

plt.tight_layout()

# ── INSPECT — adjust fig / ax before saving ──────────────
# Examples:
#   ax.set_title("Custom title", fontsize=10)
#   ax.set_ylabel("New label")
# ─────────────────────────────────────────────────────────────────

plt.show()  # inspect inline before saving

In [ ]:
# Save Fig1_StudyProtocol 
save_figure(fig, 'Fig1_StudyProtocol')

---
## Cell 10 — Figure 2: RF Duration Paired Change (HRVBF)

In [ ]:
fig, ax = plt.subplots(figsize=(W_DOUBLE, 2.9))
gb = df_part[(df_part.Group=='HRVBF')&(df_part.Phase=='Baseline')].set_index('Participant')
gi = df_part[(df_part.Group=='HRVBF')&(df_part.Phase=='Intervention')].set_index('Participant')
common = gb.index.intersection(gi.index)
bv = gb.loc[common, 'RF_duration_min'].values
iv = gi.loc[common, 'RF_duration_min'].values
for b, i in zip(bv, iv):
    ax.plot([1, 2], [b, i], color=LGT, lw=0.8, marker='o', ms=3,
            mfc='white', mec=LGT, mew=0.6, zorder=2)
for xp, vals in zip([1, 2], [bv, iv]):
    med = np.nanmedian(vals)
    q1, q3 = np.nanpercentile(vals, [25, 75])
    ax.errorbar(xp, med, yerr=[[med-q1],[q3-med]], fmt='D',
                color=BLK, ms=5.5, capsize=4, capthick=0.8, lw=1.2, zorder=5)
paired = [(b, i) for b, i in zip(bv, iv) if not (np.isnan(b) or np.isnan(i))]
bpv, ipv = [p[0] for p in paired], [p[1] for p in paired]
stat, p = wilcoxon(bpv, ipv, alternative='two-sided', zero_method='wilcox')
N = len(bpv)
Z = (stat - N*(N+1)/4) / np.sqrt(N*(N+1)*(2*N+1)/24)
r = abs(Z) / np.sqrt(N)
ax.text(1.5, max(iv)*0.95, f'W = {int(stat)},  p = {p:.3f}'+('*' if p<.05 else ''),
        ha='center', fontsize=8,
        bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='0.6', lw=0.4))
ax.set_xticks([1, 2])
ax.set_xticklabels(['Baseline', 'Paced breathing'])
ax.set_ylabel('Cardiac RF duration (min)')
ax.set_xlim(0.6, 2.4)
ax.grid(axis='y', lw=0.3, ls=':', color='0.88', zorder=0)
fig.tight_layout()

plt.tight_layout()

# ── INSPECT — adjust fig / ax before saving ──────────────
# Examples:
#   ax.set_title("Custom title", fontsize=10)
#   ax.set_ylabel("New label")
# ─────────────────────────────────────────────────────────────────

plt.show()  # inspect inline before saving

In [ ]:
# Save Fig2_RF_Duration_HRVBF
save_figure(fig, 'Fig2_RF_Duration_HRVBF')

---
## Cell 11 — Figure 3: Cardiac vs Cardiorespiratory RF Windows

In [ ]:
fig, ax = plt.subplots(figsize=(W_DOUBLE, 2.9))
bf_int = df_part[(df_part.Group=='HRVBF')&(df_part.Phase=='Intervention')].sort_values('Participant')
x = np.arange(len(bf_int))
BW = 0.38
ax.bar(x - BW/2, bf_int.n_RF_cardiac, BW, color=BLK, label='Cardiac RF')
ax.bar(x + BW/2, bf_int.n_RF_CR.fillna(0), BW, color=LGT,
       edgecolor=MID, linewidth=0.4, label='Cardiorespiratory RF')
ax.set_xticks(x)
ax.set_xticklabels(list(bf_int.Participant), fontsize=7.5)
ax.set_ylabel('Qualifying windows')
ax.set_ylim(0, bf_int.n_RF_cardiac.max() * 1.35 + 1)
ax.legend(frameon=False, fontsize=7.5, loc='upper right',
          handlelength=1.2, handleheight=0.9)
ax.grid(axis='y', lw=0.3, ls=':', color='0.88', zorder=0)
fig.tight_layout()

plt.tight_layout()

# ── INSPECT — adjust fig / ax before saving ──────────────
# Examples:
#   ax.set_title("Custom title", fontsize=10)
#   ax.set_ylabel("New label")
# ─────────────────────────────────────────────────────────────────

plt.show()  # inspect inline before saving

In [ ]:
# Save Fig3_RF_Cardiac_vs_CR — run only when satisfied
save_figure(fig, 'Fig3_RF_Cardiac_vs_CR')

---
## Cell 12 — Figure 4: Window-Level LFpeak/TP Trajectory (Exemplar)

In [ ]:
# Select HRVBF participant nearest the group median RF coverage
df_bfi = df_part[(df_part.Group=='HRVBF')&(df_part.Phase=='Intervention')]
med_cov = df_bfi.RF_percent.median()
ex_pid  = df_bfi.loc[(df_bfi.RF_percent - med_cov).abs().idxmin(), 'Participant']

fig, ax = plt.subplots(figsize=(W_SINGLE, 2.8))
sub = df_win_full[(df_win_full.Participant==ex_pid) & (df_win_full.Phase=='Intervention')]
t_mid  = sub.WindowStart_s + WIN_LEN/2
lf_tp  = sub.LFpeak_TP
rf_m   = ((sub.f_peak_Hz.between(*RF_BAND)) & (sub.LFpeak_TP >= MIN_RF_PROP)).values
ax.plot(t_mid, lf_tp, color=BLK, lw=0.9, zorder=3)
ax.scatter(t_mid[rf_m], lf_tp.values[rf_m], color=BLK, zorder=5, s=14,
           label='RF-qualifying window')
ax.axhline(MIN_RF_PROP, color='0.0', ls='--', lw=0.7, label=f'Threshold ({MIN_RF_PROP})')
ax.fill_between(t_mid, MIN_RF_PROP, lf_tp.clip(upper=1.0),
                where=rf_m, alpha=0.18, color=BLK, zorder=2)
cov_val = df_bfi.loc[df_bfi.Participant==ex_pid, 'RF_percent'].values[0]
n_rf_w  = rf_m.sum()
ax.text(0.01, 0.97,
        f'{ex_pid} — Paced breathing  |  RF coverage: {cov_val:.0f}%  ({n_rf_w}/{len(sub)} windows)',
        transform=ax.transAxes, fontsize=7.5, va='top')
ax.set_xlabel('Time into session (s)')
ax.set_ylabel('Normalised LF power  [LF/(LF+HF)]')
ax.set_ylim(-0.03, 1.06)
ax.legend(frameon=False, fontsize=7.5, loc='upper right')
ax.grid(lw=0.3, ls=':', color='0.88', zorder=0)
fig.tight_layout()

plt.tight_layout()

# ── INSPECT — adjust fig / ax before saving ──────────────
# Examples:
#   ax.set_title("Custom title", fontsize=10)
#   ax.set_ylabel("New label")
# ─────────────────────────────────────────────────────────────────

plt.show()  # inspect inline before saving

In [ ]:
# Save Fig4_Window_Trajectory — run only when satisfied
save_figure(fig, 'Fig4_Window_Trajectory')

---
## Cell 13 — Figure 5: Group x Condition LFpeak/TP (Violin + Box)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W_SINGLE, 2.9), sharey=True)
for ax, (grp, fill, panel) in zip(axes, [('HRVBF', FILL_H, 'a'), ('Control', FILL_C, 'b')]):
    data = [df_win_full[(df_win_full.Group==grp) & (df_win_full.Phase==ph)]
            ['LFpeak_TP'].dropna().values
            for ph in ['Baseline', 'Intervention']]
    vp = ax.violinplot(data, positions=[1, 2], showmedians=False, widths=0.55)
    for b in vp['bodies']:
        b.set_facecolor(fill); b.set_alpha(0.55)
        b.set_edgecolor(BLK); b.set_linewidth(0.4)
    for part in ['cbars','cmins','cmaxes']:
        vp[part].set_color(BLK); vp[part].set_linewidth(0.5)
    ax.boxplot(data, positions=[1, 2], widths=0.22, patch_artist=False,
               medianprops=dict(color=BLK, lw=1.8),
               whiskerprops=dict(color=MID, lw=0.6),
               capprops=dict(color=MID, lw=0.6),
               boxprops=dict(color=MID, lw=0.6),
               flierprops=dict(marker='o', ms=1.5, color=MID, alpha=0.5))
    ax.axhline(MIN_RF_PROP, ls='--', lw=0.7, color=BLK, label=f'RF threshold ({MIN_RF_PROP})')
    xl = 'Paced breathing' if grp == 'HRVBF' else 'Passive sitting'
    ax.set_xticks([1, 2]); ax.set_xticklabels(['Baseline', xl])
    ax.text(0.04, 0.97, f'({panel}) {grp}', transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='top')
    if grp == 'HRVBF': ax.set_ylabel('Normalised LF power  [LF/(LF+HF)]')
    ax.set_ylim(-0.02, 1.04)
    ax.grid(axis='y', lw=0.3, ls=':', color='0.88', zorder=0)
    if grp == 'HRVBF': ax.legend(frameon=False, fontsize=7.5, loc='upper right')
# LMM annotation
lr = lmm_results['full']
if lr:
    fig.text(0.5, 1.01,
             f'LMM Group x Condition: beta = {lr["beta_inter"]:.3f}, '
             f'95% CI [{lr["ci_lo_inter"]:.3f}, {lr["ci_hi_inter"]:.3f}], '
             f'p = {lr["p_inter"]:.3f};  ICC = {lr["icc"]:.3f};  N_eff = {lr["n_eff"]:.0f}',
             ha='center', fontsize=7.5, color=MID)
plt.tight_layout(w_pad=1.8)

plt.tight_layout()

# ── INSPECT — adjust fig / ax before saving ──────────────
# Examples:
#   ax.set_title("Custom title", fontsize=10)
#   ax.set_ylabel("New label")
# ─────────────────────────────────────────────────────────────────

plt.show()  # inspect inline before saving

In [ ]:
# Save Fig5_LFnorm_Group_Condition — run only when satisfied
save_figure(fig, 'Fig5_LFnorm_Group_Condition')

---
## Cell 14 — Figure 6: Cardiac RF Coverage (Both Groups)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W_SINGLE, 3.0), sharey=True)
for ax, (grp, fill, panel, lbl) in zip(axes, [
        ('HRVBF',  FILL_H, 'a', 'Paced breathing'),
        ('Control',FILL_C, 'b', 'Passive sitting')]):
    g = df_part[(df_part.Group==grp) & (df_part.Phase=='Intervention')].sort_values('Participant')
    fc_list = [fill if r else LGT for r in g.RF_achieved]
    bars = ax.bar(range(len(g)), g.RF_percent.fillna(0),
                  color=fc_list, edgecolor=BLK, linewidth=0.4, width=0.62, zorder=3)
    ax.set_xticks(range(len(g)))
    ax.set_xticklabels(list(g.Participant), fontsize=7.5)
    for bar, (_, row) in zip(bars, g.iterrows()):
        ht = bar.get_height()
        m  = 'Y' if row.RF_achieved else 'n'
        c  = BLK if row.RF_achieved else MID
        ax.text(bar.get_x() + bar.get_width()/2, ht + 1.5,
                m, ha='center', va='bottom', fontsize=8.5, color=c, fontweight='bold')
    n_rf, n = g.RF_achieved.sum(), len(g)
    lo, hi  = clopper_pearson(n_rf, n)
    ax.text(0.03, 0.97, f'({panel}) {lbl}', transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='top')
    ax.text(0.97, 0.97,
            f'RF: {n_rf}/{n}  ({100*n_rf/n:.0f}%)\n[{100*lo:.0f}\u2013{100*hi:.0f}%]',
            transform=ax.transAxes, ha='right', va='top', fontsize=7,
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='0.6', lw=0.4))
    if grp == 'HRVBF': ax.set_ylabel('Cardiac RF coverage (%)')
    ax.set_ylim(0, 115)
    ax.grid(axis='y', lw=0.3, ls=':', color='0.88', zorder=0)
leg = [mpatches.Patch(fc=FILL_H, ec=BLK, lw=0.4, label='RF achieved (Y)'),
       mpatches.Patch(fc=LGT,    ec=MID, lw=0.4, label='Not achieved (n)')]
axes[0].legend(handles=leg, frameon=False, fontsize=7, loc='lower right', handlelength=1)
plt.tight_layout(w_pad=1.8)

plt.tight_layout()

# ── INSPECT — adjust fig / ax before saving ──────────────
# Examples:
#   ax.set_title("Custom title", fontsize=10)
#   ax.set_ylabel("New label")
# ─────────────────────────────────────────────────────────────────

plt.show()  # inspect inline before saving

In [ ]:
# Save Fig6_RF_Coverage — run only when satisfied
save_figure(fig, 'Fig6_RF_Coverage')

---
## Cell 15 — Figure 7: Paired RF Duration Change (Both Groups)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W_SINGLE, 2.9), sharey=True)
for ax, (grp, fill, panel) in zip(axes, [
        ('HRVBF',   FILL_H, 'a'),
        ('Control', FILL_C, 'b')]):
    gb = df_part[(df_part.Group==grp)&(df_part.Phase=='Baseline')].set_index('Participant')
    gi = df_part[(df_part.Group==grp)&(df_part.Phase=='Intervention')].set_index('Participant')
    common = gb.index.intersection(gi.index)
    bv = gb.loc[common, 'RF_duration_min'].values
    iv = gi.loc[common, 'RF_duration_min'].values
    for b, i in zip(bv, iv):
        ax.plot([1, 2], [b, i], color=LGT, lw=0.8, marker='o', ms=3,
                mfc='white', mec=LGT, mew=0.6, zorder=2)
    for xp, vals in zip([1, 2], [bv, iv]):
        med = np.nanmedian(vals)
        q1, q3 = np.nanpercentile(vals, [25, 75])
        ax.errorbar(xp, med, yerr=[[med-q1],[q3-med]], fmt='D',
                    color=BLK, ms=5.5, capsize=4, capthick=0.8, lw=1.2, zorder=5)
    xl = 'Paced breathing' if grp == 'HRVBF' else 'Passive sitting'
    ax.set_xticks([1, 2]); ax.set_xticklabels(['Baseline', xl])
    ax.text(0.04, 0.97, f'({panel}) {grp}', transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='top')
    if grp == 'HRVBF': ax.set_ylabel('Cardiac RF duration (min)')
    ax.set_xlim(0.55, 2.45)
    ax.grid(axis='y', lw=0.3, ls=':', color='0.88', zorder=0)
    paired = [(b, i) for b, i in zip(bv, iv) if not (np.isnan(b) or np.isnan(i))]
    if len(paired) >= 3:
        bpv = [p[0] for p in paired]; ipv = [p[1] for p in paired]
        _, pv = wilcoxon(bpv, ipv, alternative='two-sided', zero_method='wilcox')
        ax.text(1.5, ax.get_ylim()[1]*0.91,
                'p = ' + f'{pv:.3f}' + ('*' if pv < .05 else ''),
                ha='center', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='0.6', lw=0.4))
plt.tight_layout(w_pad=1.8)

plt.tight_layout()

# ── INSPECT — adjust fig / ax before saving ──────────────
# Examples:
#   ax.set_title("Custom title", fontsize=10)
#   ax.set_ylabel("New label")
# ─────────────────────────────────────────────────────────────────

plt.show()  # inspect inline before saving

In [ ]:
# Save Fig7_RF_Duration_Paired — run only when satisfied
save_figure(fig, 'Fig7_RF_Duration_Paired')

---
## Cell 16 — Figure 8: Window-Level Trajectories (HRVBF vs Control Exemplar)

In [ ]:
df_cti = df_part[(df_part.Group=='Control')&(df_part.Phase=='Intervention')]
med_cc  = df_cti.RF_percent.median()
ex_ct   = df_cti.loc[(df_cti.RF_percent - med_cc).abs().idxmin(), 'Participant']

fig, axes = plt.subplots(2, 1, figsize=(W_SINGLE, 4.5), sharex=False)
for ax, (pid, grp_lbl, ph_lbl, fill, panel) in zip(axes, [
        (ex_pid, 'HRVBF',   'Paced breathing',  FILL_H, 'a'),
        (ex_ct,  'Control', 'Passive sitting',   FILL_C, 'b')]):
    sub = df_win_full[(df_win_full.Participant==pid) & (df_win_full.Phase=='Intervention')]
    if sub.empty:
        ax.text(0.5, 0.5, f'{pid} — no data', ha='center', va='center',
                transform=ax.transAxes); continue
    t_mid = sub.WindowStart_s + WIN_LEN/2
    lf_tp = sub.LFpeak_TP
    rf_m  = ((sub.f_peak_Hz.between(*RF_BAND)) & (sub.LFpeak_TP >= MIN_RF_PROP)).values
    ax.plot(t_mid, lf_tp, color=BLK, lw=0.9, zorder=3)
    ax.scatter(t_mid[rf_m], lf_tp.values[rf_m], color=BLK, zorder=5,
               s=14, label='RF-qualifying window')
    ax.axhline(MIN_RF_PROP, color='0.0', ls='--', lw=0.7, label=f'Threshold ({MIN_RF_PROP})')
    ax.fill_between(t_mid, MIN_RF_PROP, lf_tp.clip(upper=1.0),
                    where=rf_m, alpha=0.18, color=BLK, zorder=2)
    cov_v = df_part[(df_part.Participant==pid)&(df_part.Phase=='Intervention')]['RF_percent'].values[0]
    ax.text(0.01, 0.97,
            f'({panel}) {pid} \u2014 {ph_lbl}  |  RF coverage: {cov_v:.0f}%  ({rf_m.sum()}/{len(sub)} windows)',
            transform=ax.transAxes, fontsize=7.5, va='top')
    ax.set_ylabel('LF/(LF+HF)')
    if panel == 'b': ax.set_xlabel('Time into session (s)')
    ax.set_ylim(-0.03, 1.06)
    ax.legend(frameon=False, fontsize=7.5, loc='upper right')
    ax.grid(lw=0.3, ls=':', color='0.88', zorder=0)
plt.tight_layout(h_pad=2.0)

plt.tight_layout()

# ── INSPECT — adjust fig / ax before saving ──────────────
# Examples:
#   ax.set_title("Custom title", fontsize=10)
#   ax.set_ylabel("New label")
# ─────────────────────────────────────────────────────────────────

plt.show()  # inspect inline before saving

In [ ]:
# Save Fig8_Exemplar_Trajectories — run only when satisfied
save_figure(fig, 'Fig8_Exemplar_Trajectories')

---
## Cell 17 — Figure 9: Multi-Epoch Beta Gradient

In [ ]:
# ════════════════════════════════════════════════════════════════════
# FIGURE 9 — Multi-epoch beta gradient
# ════════════════════════════════════════════════════════════════════
#
# ── EDIT ANYTHING BELOW THIS LINE before running ────────────────────

# --- Figure dimensions (inches) ------------------------------------
FIG_WIDTH   = W_DOUBLE     # 84 mm = 3.31 in  (use W_SINGLE for wider)
FIG_HEIGHT  = 3.4          # inches

# --- Bar chart layout -------------------------------------------
BAR_WIDTH   = 0.52         # bar width in data units
Y_MIN       = -0.005       # y-axis lower limit
Y_PAD_TOP   = 0.07         # extra headroom above tallest CI cap

# --- Colors ---------------------------------------------------
COLOR_SIG   = BLUE         # bar fill when p < alpha_thresh
COLOR_NSIG  = LGT          # bar fill when p >= alpha_thresh
COLOR_EC    = BLK          # bar edge colour (all bars)
COLOR_CI    = BLK          # CI whisker colour
COLOR_ARROW = AMBER        # front-loaded gradient arrow colour
COLOR_GRID  = '0.88'       # horizontal grid colour

# --- Significance threshold ------------------------------------
ALPHA_THRESH = 0.05

# --- CI display -----------------------------------------------
SHOW_CI_UPPER = True       # show upper CI whisker (above bar top)
SHOW_CI_LOWER = False      # show lower CI whisker (below bar top) — set True to restore
CI_CAPSIZE    = 3.5        # cap tick length
CI_LW         = 0.8        # CI line width

# --- Beta labels inside bars ----------------------------------
SHOW_BETA_LABELS  = True   # show beta value inside each bar
BETA_LABEL_FS     = 7      # font size for beta labels

# --- Significance star above CI cap ---------------------------
SHOW_SIG_STAR     = True
SIG_STAR_FS       = 12
SIG_STAR_GAP      = 0.007  # gap above CI cap (data units)

# --- Front-loaded gradient arrow ------------------------------
SHOW_ARROW        = True
ARROW_LABEL       = 'Front-loaded effect'
ARROW_LABEL_FS    = 7.5
ARROW_LABEL_STYLE = 'italic'  # 'normal' or 'italic'
ARROW_GAP         = 0.030     # above CI cap in data units
ARROW_LABEL_GAP   = 0.015    # above arrow start

# --- Axis labels ----------------------------------------------
Y_LABEL  = 'Group \u00d7 Condition interaction (\u03b2)'
X_LABEL  = ''              # leave empty for no x-axis label

# --- X-tick epoch labels (two lines) -------------------------
EP_LABELS = ['Full\nsession', 'First\n5 min', 'Centre\n5 min', 'Last\n5 min']

# --- Legend ---------------------------------------------------
SHOW_LEGEND  = True
LEGEND_LOC   = 'upper right'
LEGEND_FS    = 7.5
LEG_LABEL_SIG  = 'p < .05'
LEG_LABEL_NSIG = 'p \u2265 .05'

# ── DO NOT EDIT BELOW (data wrangling) ──────────────────────────────
ep_keys   = list(EPOCHS.keys())
betas     = [lmm_results[ek]['beta_inter']    for ek in ep_keys]
ci_lo_err = [lmm_results[ek]['beta_inter'] - lmm_results[ek]['ci_lo_inter'] for ek in ep_keys]
ci_hi_err = [lmm_results[ek]['ci_hi_inter'] - lmm_results[ek]['beta_inter'] for ek in ep_keys]
pvals     = [lmm_results[ek]['p_inter']       for ek in ep_keys]

# ── BUILD FIGURE ────────────────────────────────────────────────────
fig9, ax9 = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))
x = np.arange(len(ep_keys))

# Bars
bar_colors = [COLOR_SIG if p < ALPHA_THRESH else COLOR_NSIG for p in pvals]
ax9.bar(x, betas, BAR_WIDTH, color=bar_colors, edgecolor=COLOR_EC,
        linewidth=0.5, zorder=3)

# CI whiskers — upper only
for xi, b, p, cle, chi in zip(x, betas, pvals, ci_lo_err, ci_hi_err):
    bar_top = b           # top of bar = beta value
    cap_hi  = b + chi     # upper CI cap
    if SHOW_CI_UPPER:
        # Vertical line from bar top to upper CI cap
        ax9.plot([xi, xi], [bar_top, cap_hi], color=COLOR_CI, lw=CI_LW, zorder=4)
        # Upper cap tick
        ax9.plot([xi - 0.15, xi + 0.15], [cap_hi, cap_hi],
                 color=COLOR_CI, lw=CI_LW, zorder=4)
    if SHOW_CI_LOWER:
        cap_lo = b - cle  # lower CI cap
        ax9.plot([xi, xi], [max(cap_lo, 0), bar_top], color=COLOR_CI, lw=CI_LW, zorder=4)
        ax9.plot([xi - 0.15, xi + 0.15], [cap_lo, cap_lo],
                 color=COLOR_CI, lw=CI_LW, zorder=4)

# Beta value labels centred inside bars
if SHOW_BETA_LABELS:
    for xi, b, p in zip(x, betas, pvals):
        if b > 0.008:  # only if bar is tall enough to label
            ax9.text(xi, b / 2, f'{b:.3f}',
                     ha='center', va='center', fontsize=BETA_LABEL_FS,
                     color='white' if p < ALPHA_THRESH else BLK,
                     fontweight='bold', zorder=5)

# Significance stars above upper CI cap
if SHOW_SIG_STAR:
    for xi, b, p, chi in zip(x, betas, pvals, ci_hi_err):
        if p < ALPHA_THRESH:
            ax9.text(xi, b + chi + SIG_STAR_GAP, '*',
                     ha='center', va='bottom', fontsize=SIG_STAR_FS,
                     color=COLOR_SIG, fontweight='bold', zorder=6)

# Front-loaded gradient arrow
if SHOW_ARROW:
    arrow_y1 = betas[1] + ci_hi_err[1] + ARROW_GAP
    arrow_y3 = betas[3] + ci_hi_err[3] + ARROW_GAP
    ax9.annotate('', xy=(x[3], arrow_y3), xytext=(x[1], arrow_y1),
                 arrowprops=dict(arrowstyle='-|>', color=COLOR_ARROW, lw=0.8,
                                 connectionstyle='arc3,rad=-0.2'))
    ax9.text(2.5, arrow_y1 + ARROW_LABEL_GAP, ARROW_LABEL,
             ha='center', va='bottom', fontsize=ARROW_LABEL_FS,
             color=COLOR_ARROW, style=ARROW_LABEL_STYLE)

# Axes
ax9.axhline(0, color=BLK, lw=0.5)
ax9.set_xticks(x)
ax9.set_xticklabels(EP_LABELS)
# Bold+colour the significant epoch x-label
for tick, p in zip(ax9.get_xticklabels(), pvals):
    if p < ALPHA_THRESH:
        tick.set_color(COLOR_SIG); tick.set_fontweight('bold')
if X_LABEL:
    ax9.set_xlabel(X_LABEL)
ax9.set_ylabel(Y_LABEL)
y_top = max(b + hi for b, hi in zip(betas, ci_hi_err)) + Y_PAD_TOP
ax9.set_ylim(Y_MIN, y_top)
ax9.grid(axis='y', lw=0.3, ls=':', color=COLOR_GRID, zorder=0)

# Legend
if SHOW_LEGEND:
    leg = [mpatches.Patch(fc=COLOR_SIG,  ec=COLOR_EC, lw=0.4, label=LEG_LABEL_SIG),
           mpatches.Patch(fc=COLOR_NSIG, ec=MID,      lw=0.4, label=LEG_LABEL_NSIG)]
    ax9.legend(handles=leg, frameon=False, fontsize=LEGEND_FS,
               loc=LEGEND_LOC, handlelength=1.2)

plt.tight_layout()

# ── INSPECT — make any further changes to fig9 / ax9 here ──────────
# Examples:
#   ax9.set_title('My custom title', fontsize=10)
#   ax9.set_ylim(-0.01, 0.25)
#   fig9.set_size_inches(W_SINGLE, 3.8)
#   ax9.get_xticklabels()[0].set_text('All windows')
# ───────────────────────────────────────────────────────────────────

plt.show()    # <-- renders inline in Jupyter for inspection
print('\nFig 9 rendered. Edit the variables above and re-run to adjust.')
print('Run the SAVE cell below only when satisfied.')